# Role Recognition with Vertex AI Gemini

This notebook demonstrates how to use the `RoleRecognizer` class, a Python module designed to identify roles (AGENT or CUSTOMER) within conversation transcripts using Vertex AI's Generative Models (Gemini).

## 1. Setup and Installation

First, we need to install the necessary libraries and set up authentication for Vertex AI.

In [ ]:
# Install necessary libraries
!pip install -q google-generativeai google-cloud-aiplatform

import os
import json
import pathlib

# Import Vertex AI components after installation
import vertexai
from vertexai.generative_models import GenerationConfig, GenerativeModel

### 1.1. Google Cloud Authentication

You'll need a Google Cloud project and an API key with access to Vertex AI. Replace the placeholders with your actual project ID, location, and API key.

**Note:** For Colab, you might use `google.colab.auth.authenticate_user()` if you prefer to authenticate via a browser flow instead of an API key. For this example, we'll use `GOOGLE_API_KEY` for simplicity.

In [ ]:
# --- IMPORTANT: Replace with your actual values ---
PROJECT_ID = "YOUR_PROJECT_ID"  # e.g., "my-gcp-project-12345"
LOCATION = "us-central1"      # e.g., "us-central1"
GOOGLE_API_KEY = "YOUR_API_KEY" # Your Google Cloud API Key for Vertex AI
# --------------------------------------------------

os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY

# Initialize Vertex AI
try:
    vertexai.init(project=PROJECT_ID, location=LOCATION)
    print(f"Vertex AI initialized for project '{PROJECT_ID}' in location '{LOCATION}'.")
except Exception as e:
    print(f"Error initializing Vertex AI: {e}")
    print("Please ensure your PROJECT_ID, LOCATION, and GOOGLE_API_KEY are correct and have the necessary permissions.")

## 2. The `RoleRecognizer` Module

Below is the source code for the `role_recognizer.py` module. We'll write this to a file and then import the `RoleRecognizer` class.

In [ ]:
%%writefile role_recognizer.py
# Copyright 2025 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

""" Role recognition class for conversation transcripts. """

import json
import pathlib
from vertexai.generative_models import GenerationConfig, GenerativeModel


class RoleRecognizer:
    """
    Role recognition class for conversation transcripts.
    """

    # _SCHEMA_DIR = pathlib.Path(__file__).parent.parent / "common/utils/schemas/" # This path is relative and won't work in this standalone example
    _RESPONSE_MIME_TYPE = "application/json"
    _TEMPERATURE = 0
    _TOP_P = 0.95
    _TOP_K = 40
    _DEFAULT_PROMPT = """
        You are a multilingual role recognition expert in identifying roles within a conversation transcript.

        Task:
        Identify the role (AGENT or CUSTOMER) of each utterance in the given conversation.
        If the conversation involves only two agents, assign one as the CUSTOMER and the other as the AGENT to maintain role differentiation.

        **Guidelines:**

        * Analyze the entire conversation to understand the context.
        * Assign the correct role to each line of the utterance based on its content and the flow of the conversation.
        * DO NOT add or split, modify or combine any utterances and sentences.

        **Important Considerations:**
            * **Agent-to-Agent Conversations:**
            In conversations between two agents, designate one as "AGENT" and the other as "CUSTOMER" based on the context of their interaction.
            Consider which agent is primarily providing information or support (AGENT) and which is primarily receiving it (CUSTOMER).
            The distinction can be subtle, so focus on the flow of information and assistance.

        **Few-Shot Examples:**

        **Example 1 (Agent-to-Customer):**
            *Input:*
                Utterance 1: "Hello, how can I help you today?"
                Utterance 2: "Hi, I'm having trouble logging in."
                Utterance 3: "Have you tried resetting your password?"
                Utterance 4: "Yes, but it's not working."
                Utterance 5: "Okay, let's try a different approach."
            *Output:*
                Utterance 1: "AGENT"
                Utterance 2: "CUSTOMER"
                Utterance 3: "AGENT"
                Utterance 4: "CUSTOMER"
                Utterance 5: "AGENT"

        **Example 2 (Agent-to-Agent):**
            *Input:*
                Utterance 1: "Hey, I'm having trouble with this customer's issue.  They're saying the order hasn't shipped yet."
                Utterance 2: "Okay, let me check the order status for you. What's the order number?"
                Utterance 3: "It's #12345."
                Utterance 4: "Thanks.  Looks like there was a delay at the warehouse. I'll update the customer and offer a discount."
            *Output:*
                Utterance 1: "CUSTOMER"
                Utterance 2: "AGENT"
                Utterance 3: "CUSTOMER"
                Utterance 4: "AGENT"

        Conversation:
        {conversation}
    """
    _RESPONSE_SCHEMA = {
        "type": "object",
        "properties": {
            "predictions": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "role": {
                            "type": "string",
                            "enum": [
                                "AGENT",
                                "CUSTOMER"
                            ]
                        },
                        "uid": {
                            "type": "integer"
                        }
                    },
                    "required": [
                        "role",
                        "uid"
                    ]
                }
            }
        },
        "required": [
            "predictions"
        ]
    }

    def __init__(self, model_name: str = "gemini-2.5-pro", prompt: str = ""):
        self.model_name = model_name
        self.prompt = prompt

        self.model = GenerativeModel(model_name=self.model_name)
        self.generation_config = GenerationConfig(
            temperature=self._TEMPERATURE,
            top_p=self._TOP_P,
            top_k=self._TOP_K,
            response_mime_type=self._RESPONSE_MIME_TYPE,
            response_schema=self._RESPONSE_SCHEMA,
        )

        if not self.prompt:
            self.prompt = self._DEFAULT_PROMPT

    def predict_roles(self, conversation: dict) -> dict:
        """Predicts roles in a conversation transcript.

        Args:
            conversation (dict): Conversation text/utterances along with utterance ID

        Returns:
            roles (dict): Roles assigned to each utterance
        """

        response = self.model.generate_content(
            self.prompt.format(conversation=conversation),
            generation_config=self.generation_config,
            stream=False,
        )
        return json.loads(response.text)

    def combine(self, conversation: dict, roles: dict) -> str:
        """Combines the original conversations with the roles predicted
        Args:
            conversation (dict): Conversation text/utterances along with utterance ID
            roles (dict): Roles assigned to each utterance
        Returns:
            converstion (str): Will return the conversation in a json (str) file format

        """
        for index, result in enumerate(conversation["results"]):
            try:
                if roles["predictions"][index]["role"] == "AGENT":
                    result["alternatives"][0]["channelTag"] = 2
                    result["channelTag"] = 2
                if roles["predictions"][index]["role"] == "CUSTOMER":
                    result["alternatives"][0]["channelTag"] = 1
                    result["channelTag"] = 1
            except (IndexError, KeyError):
                # if error defaults to customer
                result["alternatives"][0]["channelTag"] = 1
                result["channelTag"] = 1

        return json.dumps(conversation)

In [ ]:
# Now import the RoleRecognizer class from the created file
from role_recognizer import RoleRecognizer

### 2.1. `RoleRecognizer` Overview

The `RoleRecognizer` class leverages a Generative Model (defaulting to `gemini-2.5-pro`) to perform role recognition based on a provided prompt and conversation transcript. It features:

-   **`__init__(self, model_name: str = "gemini-2.5-pro", prompt: str = "")`**: Initializes the recognizer. You can specify the model name and provide a custom prompt. If no prompt is given, a default prompt optimized for role recognition is used.
-   **`predict_roles(self, conversation: dict) -> dict`**: Takes a dictionary representing a conversation and returns a dictionary with predicted roles for each utterance. The input `conversation` dictionary is stringified and passed to the model.
-   **`combine(self, conversation: dict, roles: dict) -> str`**: Merges the predicted roles back into the original conversation structure, assigning `channelTag` (1 for CUSTOMER, 2 for AGENT) to each utterance, and returns the updated conversation as a JSON string.

## 3. Demonstrating `RoleRecognizer`

Let's instantiate the `RoleRecognizer` and try it with some example conversations.

In [ ]:
# Initialize the RoleRecognizer with the default model and prompt
recognizer = RoleRecognizer()
print(f"RoleRecognizer initialized using model: {recognizer.model_name}")

### 3.1. Example 1: Agent-to-Customer Conversation

We'll start with a typical support conversation where a customer interacts with an agent.

In [ ]:
conversation_data_1 = {
    "results": [
        {
            "uid": 1,
            "alternatives": [
                {"transcript": "Hello, thank you for calling support. How can I assist you today?"}
            ]
        },
        {
            "uid": 2,
            "alternatives": [
                {"transcript": "Hi, I'm having trouble with my internet connection. It keeps dropping out."}
            ]
        },
        {
            "uid": 3,
            "alternatives": [
                {"transcript": "I understand. Have you tried restarting your router and modem?"}
            ]
        },
        {
            "uid": 4,
            "alternatives": [
                {"transcript": "Yes, I've done that several times, but it doesn't seem to help."}
            ]
        },
        {
            "uid": 5,
            "alternatives": [
                {"transcript": "Okay, let's run some diagnostics from our end. Could you confirm your account number for me?"}
            ]
        }
    ]
}

print("--- Original Conversation Data (Example 1) ---")
print(json.dumps(conversation_data_1, indent=2))

# Predict roles
print("\n--- Predicting Roles... ---")
roles_1 = recognizer.predict_roles(conversation_data_1)
print(json.dumps(roles_1, indent=2))

# Combine roles with the original conversation
# Make a copy of the conversation data as the combine method modifies it in place
combined_conversation_1 = recognizer.combine(conversation_data_1.copy(), roles_1)
print("\n--- Combined Conversation with Roles (Example 1) ---")
print(combined_conversation_1)

### 3.2. Example 2: Agent-to-Agent Conversation

The `RoleRecognizer` is designed to handle agent-to-agent interactions by assigning one as 'AGENT' and the other as 'CUSTOMER' based on who is providing/receiving primary support in that specific interaction.

In [ ]:
conversation_data_2 = {
    "results": [
        {
            "uid": 1,
            "alternatives": [
                {"transcript": "Hey Alex, I'm having trouble with this customer's issue. They're saying the order hasn't shipped yet."}
            ]
        },
        {
            "uid": 2,
            "alternatives": [
                {"transcript": "Okay, let me check the order status for you. What's the order number?"}
            ]
        },
        {
            "uid": 3,
            "alternatives": [
                {"transcript": "It's #A1B2C3D4."}
            ]
        },
        {
            "uid": 4,
            "alternatives": [
                {"transcript": "Thanks. Looks like there was a delay at the warehouse. I'll update the customer and offer a discount from my side."}
            ]
        }
    ]
}

print("--- Original Conversation Data (Example 2) ---")
print(json.dumps(conversation_data_2, indent=2))

# Predict roles
print("\n--- Predicting Roles... ---")
roles_2 = recognizer.predict_roles(conversation_data_2)
print(json.dumps(roles_2, indent=2))

# Combine roles with the original conversation
combined_conversation_2 = recognizer.combine(conversation_data_2.copy(), roles_2)
print("\n--- Combined Conversation with Roles (Example 2) ---")
print(combined_conversation_2)

## 4. Customizing the Prompt

You can also provide a custom prompt to the `RoleRecognizer` during initialization if you need to guide the model with specific instructions or few-shot examples tailored to your use case.

In [ ]:
custom_prompt = """
You are an expert at classifying speakers in a debate. 
Identify if each utterance is from the 'PRO' side or 'CON' side of the debate.
The debaters are arguing about the merits of remote work.

**Few-Shot Example:**
Input:
  Utterance 1: "Remote work offers unparalleled flexibility and productivity for many roles."
  Utterance 2: "But it often leads to communication breakdowns and a loss of company culture."
Output:
  Utterance 1: "PRO"
  Utterance 2: "CON"

Conversation:
{conversation}
"""

# Note: The response schema is hardcoded in RoleRecognizer to AGENT/CUSTOMER.
# For a truly custom role classification, you might need to modify the RoleRecognizer class's _RESPONSE_SCHEMA
# or parse the response differently if the model can ignore the schema. 
# For demonstration, we'll see if the model's freeform text can provide the custom roles.
# However, due to `response_schema`, the output will still try to fit into AGENT/CUSTOMER.
# A better approach for custom roles would be to define a new class or modify the schema within RoleRecognizer init.
# For this module, we will stick to the AGENT/CUSTOMER constraint as per the class's design.

print("The RoleRecognizer's underlying schema forces 'AGENT'/'CUSTOMER' roles.")
print("To truly customize roles beyond AGENT/CUSTOMER, the `_RESPONSE_SCHEMA` within the `RoleRecognizer` class would need modification.")
print("This example primarily shows prompt customization, but the output will still map to the defined schema roles.")

conversation_data_3 = {
    "results": [
        {
            "uid": 1,
            "alternatives": [
                {"transcript": "I believe remote work significantly reduces overhead costs for businesses."}
            ]
        },
        {
            "uid": 2,
            "alternatives": [
                {"transcript": "However, it can complicate team building and mentorship for new hires."}
            ]
        }
    ]
}

# We'll use the default recognizer for now, as the schema constrains output
# If the schema were dynamic, this would be the place to pass `custom_prompt`

print("\n--- Predicting Roles with the default recognizer (still constrained by schema) ---")
roles_3 = recognizer.predict_roles(conversation_data_3)
print(json.dumps(roles_3, indent=2))

combined_conversation_3 = recognizer.combine(conversation_data_3.copy(), roles_3)
print("\n--- Combined Conversation with Roles (Custom Prompt Scenario) ---")
print(combined_conversation_3)

## 5. Conclusion

This notebook demonstrated the `RoleRecognizer` class for identifying AGENT and CUSTOMER roles in conversation transcripts using Vertex AI's Gemini models. You learned how to:

-   Set up your environment and authenticate with Google Cloud.
-   Integrate the `role_recognizer.py` module into your workflow.
-   Use `predict_roles` to obtain role predictions.
-   Use `combine` to merge predicted roles back into the original conversation structure.
-   Understand how the model handles different conversation scenarios, including agent-to-agent interactions.

The `RoleRecognizer` provides a robust and flexible solution for automatically tagging speaker roles in conversational data, which is crucial for downstream analytics and processing in customer support and other dialogue systems.